In [2]:
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy as sp
import sklearn

sys.version, np.__version__, pd.__version__, matplotlib.__version__, sp.__version__, sklearn.__version__

('3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]',
 '2.5.4',
 '3.1.1',
 '3.11.2',
 '1.18.1',
 '1.9.1')

### Работа с dataframe
Для удобства все месяца объеденю в один столбец с помощью функции melt. 

In [3]:
train = pd.read_csv('E:/kaggle_mamo/data/train.csv')

month_cols = [c for c in train.columns if c.startswith('20')]
#id_vars - какие столбцы оставить без имзменений, value_vars столбцы которые нужно превратить в строки, var_name имя нового столбца, value_name имя нового столбца для значений
train_long = train.melt(id_vars = ['sku', 'unit'], value_vars = month_cols, var_name = 'month', value_name = 'sales')
train_long.head()

,sku,unit,month,sales
0,0,ШТ,2019-01,128.0
1,1,ШТ,2019-01,65.0
2,2,ШТ,2019-01,157.0
3,3,ШТ,2019-01,2450.0
4,4,ШТ,2019-01,119.0


### Функция скользящей валидации split_data
Для оценки качества моделей нельзя использовать случайное разбиение данных. Нужно обучаться на прошлом, а проверяться на будущем. 

Функция split_data принимает дату отсечения cutoff_date и делит данные на две части:
1. train_df - все данные до cutoff_date включительно (для обучения).
2. valid_df - ровно 3 следующих месяца после cutoff_date (для проверки).

In [17]:
def split_data(data, cutoff_date):
    #создаю копии, чтобы потом ничего не поломалось
    train_df = data[data['month'] <= cutoff_date].copy()

    valid_df = data[data['month'] > cutoff_date].copy()

    valid_months = sorted(valid_df['month'].unique())[:3]

    valid_df = valid_df[valid_df['month'].isin(valid_months)].copy()

    return train_df, valid_df

In [18]:
#проверяю, что функция работает
train_part, valid_part = split_data(train_long, '2021-08')

print("Месяцы в валидации:", valid_part['month'].unique())
print("Размер train:", train_part.shape)
print("Размер valid:", valid_part.shape)

Месяцы в валидации: <StringArray>
['2021-09', '2021-10', '2021-11']
Length: 3, dtype: str
Размер train: (52992, 4)
Размер valid: (4968, 4)


### Метрика WAPE

Функция wape реализует формулу: сумма абсолютных ошибок / сумма фактических продаж. 

In [5]:
def wape(actual, prediction):
    #работаю с сериями
    numerator = (actual - prediction).abs().sum()
    denominator = actual.abs().sum()
    return numerator / denominator

In [6]:
#проверяю что функция работает
test_actual = pd.Series([100, 10])
test_pred = pd.Series([90, 15])

result = wape(test_actual, test_pred)
print("WAPE:", round(result,3))

WAPE: 0.136


### Baseline 1: наивный прогноз
Прогноз на сентябрь-ноябрь 2021 года равен фактическим продажам за аналогичные месяцы 2020 года. 

In [31]:
naive_df = train_part[train_part['month'].isin(['2020-09', '2020-10', '2020-11'])].copy()

naive_df['month'] = naive_df['month'].replace({'2020-09' : '2021-09', '2020-10' : '2021-10', '2020-11' : '2021-11'})

naive_df = naive_df.rename(columns={'sales': 'prediction'})

naive_df = naive_df[['sku', 'month', 'prediction']]

valid_naive_df = valid_part.merge(naive_df, on = ['sku', 'month'], how = 'left')

valid_naive_df.head()

,sku,unit,month,sales,prediction
0,0,ШТ,2021-09,318.0,221.0
1,1,ШТ,2021-09,100.0,108.0
2,2,ШТ,2021-09,293.0,251.0
3,3,ШТ,2021-09,2187.0,2314.0
4,4,ШТ,2021-09,321.0,345.0


In [32]:
score_1 = wape(valid_naive_df['sales'], valid_naive_df['prediction'])

print(score_1)

0.3110627317553807


### Baseline 2: среднее за 12 месяцев
Прогноз на 3 месяца вперед это среднее арифметическое продаж за последние 12 месяцев. 

In [33]:
mean_df = train_part[train_part['month'] >= '2020-09'].copy()

mean_df = mean_df.groupby('sku')['sales'].mean().reset_index()

mean_df = mean_df.rename(columns={'sales' : 'prediction'})

valid_mean_df = valid_part.merge(mean_df, on = ['sku'], how = 'left')

valid_mean_df.head()

,sku,unit,month,sales,prediction
0,0,ШТ,2021-09,318.0,228.666667
1,1,ШТ,2021-09,100.0,78.583333
2,2,ШТ,2021-09,293.0,357.000000
3,3,ШТ,2021-09,2187.0,2211.500000
4,4,ШТ,2021-09,321.0,425.000000


In [34]:
score_2 = wape(valid_mean_df['sales'], valid_mean_df['prediction'])

print(score_2)

0.2753317497173849


### Baseline 3: последнее значение 
Берем фактический объем продаж за последний известный месяц и предполагаем, что в сентябре, октябре и ноябре продажи будут такими же.

In [24]:
last_month_df = train_part[train_part['month'] == '2021-08'].copy()

last_month_df = last_month_df.rename(columns={'sales' : 'prediction'})

valid_pred_df = valid_part.merge(last_month_df, on = 'sku', how = 'left')

valid_pred_df.head()

,sku,unit_x,month_x,sales,unit_y,month_y,prediction
0,0,ШТ,2021-09,318.0,ШТ,2021-08,121.0
1,1,ШТ,2021-09,100.0,ШТ,2021-08,75.0
2,2,ШТ,2021-09,293.0,ШТ,2021-08,404.0
3,3,ШТ,2021-09,2187.0,ШТ,2021-08,1821.0
4,4,ШТ,2021-09,321.0,ШТ,2021-08,365.0


In [25]:
score_3 = wape(valid_pred_df['sales'], valid_pred_df['prediction'])
print(score_3)

0.3804460452580045


### Вывод по базовым прогнозам (Baselines)
В ходе скользящей валидации были получены следующие значения метрики WAPE:
1. Прогноз «Последнее значение»: 0.380
2. Прогноз «Наивный»: 0.311
3. Прогноз «Среднее за 12 месяцев»: **0.275**

Наилучший результат показал прогноз по среднему значению за последние 12 месяцев. Это будет основная планка. Задача модели показать WAPE ниже 0.275.